<a href="https://colab.research.google.com/github/F1ameX/2025-ODS-NLP/blob/main/practice_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and Import Libraries

In [1]:
!pip install --quiet catboost
!pip install --quiet gensim
!pip install --quiet nltk
!pip install --quiet pymystem3

In [2]:
import os
import re
import pandas as pd
import numpy as np

import nltk
from catboost import Pool, CatBoostClassifier
from pymystem3 import Mystem

## Read Data

In [3]:
path = '/content/drive/MyDrive/ods-nlp_src/practice_2/'
train_data = pd.read_csv(os.path.join(path, 'train.csv'))
test_data = pd.read_csv(os.path.join(path, 'test.csv'))
print(f'Number of rows and columns in the train data set: {train_data.shape}')
print(f'Number of rows and columns in the test data set: {test_data.shape}')
train_data.head()

Number of rows and columns in the train data set: (48665, 2)
Number of rows and columns in the test data set: (12167, 2)


,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


In [4]:
train_data.groupby('rate').describe()

text                               
      count unique                top freq
rate                                      
1      4138   4130             Грязно    3
2      2410   2407  Отстойный магазин    2
3      6126   6070          Нормально    7
4      9922   9763               Норм   14
5     26069  24804    Хороший магазин  107

## Preparing the data and creating Catboost model

In [5]:
train_data['text'].head(15)

,text
0,Очень понравилось. Были в начале марта с соба...
1,В целом магазин устраивает.\nАссортимент позво...
2,"Очень хорошо что открылась 5 ка, теперь не над..."
3,Пятёрочка громко объявила о том как она заботи...
4,"Тесно, вечная сутолока, между рядами трудно ра..."
5,Магазин в пешей доступности. После ремонта и р...
6,Магазин хороший цены и скидки нормальные токо ...
7,"Редко сюда забегаю. Маленький магазинчик, но э..."
8,Сложно найти в торговом центре. А магазин - норм)
9,После ремонта магазин в нутри стал ещё лучше. ...


In [6]:
def process_data(df):
    df['text'] = df['text'].str.lower()
    df['text'] = df['text'].apply(lambda x: re.sub(r'([,.!?;])', r' \1 ', x))
    df['text'] = df['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))
    df['text'] = df['text'].apply(lambda x: re.sub(r'[\d]', '', x))

    mystem = Mystem()
    df['text'] = df['text'].apply(lambda x: ' '.join(mystem.lemmatize(x)))
    df['text'] = df['text'].apply(lambda x: re.sub(r'[\n]', ' ', x))
    return df

In [7]:
train_data = process_data(train_data)
test_data = process_data(test_data)

In [8]:
train_data.head(15)

,rate,text
0,4,очень понравиться быть в начало ма...
1,5,в целое магазин устраивать ассортиме...
2,5,очень хорошо что открываться ка т...
3,3,пятерочка громко объявлять о то как ...
4,3,тесно вечный сутолока между ряд ...
5,4,магазин в пеший доступность после ...
6,5,магазин хороший цена и скидка нормал...
7,3,редко сюда забегать маленький магази...
8,5,сложно находить в торговый центр а...
9,4,после ремонт магазин в нутри станови...


In [10]:
X_train = train_data['text']
y_train = train_data['rate']

X_test = test_data['text']


model = CatBoostClassifier(
    iterations = 250,
    depth = 6,
    random_seed = 52,
)

model.fit(
    X_train,
    y_train,
    verbose=True,
    text_features = [0]
)

Learning rate set to 0.311252
0:	learn: 1.2260014	total: 3.01s	remaining: 12m 28s
1:	learn: 1.1069761	total: 5.1s	remaining: 10m 32s
2:	learn: 1.0418000	total: 6.92s	remaining: 9m 29s
3:	learn: 0.9969966	total: 8.76s	remaining: 8m 58s
4:	learn: 0.9665840	total: 10.4s	remaining: 8m 30s
5:	learn: 0.9490324	total: 12.2s	remaining: 8m 14s
6:	learn: 0.9351377	total: 14.1s	remaining: 8m 8s
7:	learn: 0.9254564	total: 16.6s	remaining: 8m 22s
8:	learn: 0.9171540	total: 18.9s	remaining: 8m 25s
9:	learn: 0.9120253	total: 20.5s	remaining: 8m 10s
10:	learn: 0.9080493	total: 22.1s	remaining: 7m 59s
11:	learn: 0.9029516	total: 23.7s	remaining: 7m 50s
12:	learn: 0.9013978	total: 25.3s	remaining: 7m 40s
13:	learn: 0.8993087	total: 26.8s	remaining: 7m 32s
14:	learn: 0.8971553	total: 28.6s	remaining: 7m 28s
15:	learn: 0.8956294	total: 31.1s	remaining: 7m 35s
16:	learn: 0.8913729	total: 33.4s	remaining: 7m 37s
17:	learn: 0.8899923	total: 34.9s	remaining: 7m 29s
18:	learn: 0.8889738	total: 36.4s	remaining:

## Predict

In [11]:
dataset_test = Pool(
    data = X_test,
    text_features = [0]
)

predict_classes = model.predict(dataset_test)
predictions = predict_classes

## Create submission

In [12]:
sample_submission = pd.read_csv(os.path.join(path, 'sample_submission.csv'))
sample_submission['rate'] = predictions
sample_submission.head()

,index,rate
0,0,5
1,1,5
2,2,5
3,3,4
4,4,1


In [13]:
sample_submission.to_csv(os.path.join(path, 'submission.csv'), index=False)